In [11]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 20)

path_csv = Path("..") / "data" / "raw" / "online_retail_II_gabungan.csv"
print("CSV ada:", path_csv.exists())

gabungan = pd.read_csv(
    path_csv,
    dtype={"Invoice": "string", "StockCode": "string"},
    parse_dates=["InvoiceDate"],
)
print(gabungan.shape)

CSV ada: True
(1044848, 8)


In [12]:
bukan5= gabungan.loc[~gabungan["StockCode"].str.match(r"^\d{5}", na=False)]
ringkas = (bukan5.groupby("StockCode")
           .agg(baris=("Invoice", "size"),
                       contih_deskripsi=("Description", "first"),
                       harga_rata2=("Price", "mean"))
                       .sort_values("baris", ascending=False))
print("Jumlah kode berbeda:", len(ringkas))
ringkas.head(40)

                       

Jumlah kode berbeda: 62


,baris,contih_deskripsi,harga_rata2
StockCode,,,
POST,2086,POSTAGE,32.849128
DOT,1425,DOTCOM POSTAGE,217.448505
M,1398,Manual,536.074077
C2,277,CARRIAGE,49.451264
D,173,Discount,71.055491
S,102,SAMPLES,61.515196
BANK CHARGES,100,Bank Charges,365.111310
ADJUST,67,Adjustment by john on 26/01/2010 16,163.591343
AMAZONFEE,36,AMAZON FEE,7290.447222


In [13]:
neg = gabungan["Quantity"] <= 0
batal = gabungan["Invoice"].str.startswith("C")
print(pd.crosstab(neg, batal, rownames=["Qty<=0"], colnames=["Invoice C]"]))

neg_tanpa_c = gabungan[neg & ~batal]
print("Qty<=0 tanpa awalan C:", len(neg_tanpa_c))
print("  yang harganya nol:", f"{(neg_tanpa_c["Price"] == 0).mean():.1%}")
print("  yang tanpa Customer ID:", f"{neg_tanpa_c['Customer ID'].isna().mean():.1%}")

Invoice C]    False  True 
Qty<=0                    
False       1022290      1
True           3393  19164
Qty<=0 tanpa awalan C: 3393
  yang harganya nol: 100.0%
  yang tanpa Customer ID: 100.0%


In [14]:
kembar = gabungan[gabungan.duplicated(keep="first")]
print("Baris kembar:", len(kembar))
print("Pendapatan di dalamnya:", round((kembar["Quantity"] * kembar["Price"]).sum(), 2))
print("Punya Customer ID:", f"{kembar['Customer ID'].notna().mean():.1%}")

Baris kembar: 11812
Pendapatan di dalamnya: 54228.42
Punya Customer ID: 98.8%


In [15]:
ringkas.tail(22)

,baris,contih_deskripsi,harga_rata2
StockCode,,,
DCGS0036,1,NaN,0.00
DCGS0044,1,HANDZ-OFF CAR FRESHENER,2.57
C3,1,NaN,0.00
DCGS0006,1,NaN,0.00
DCGS0016,1,NaN,0.00
DCGS0027,1,NaN,0.00
DCGS0067,1,ebay,0.00
DCGS0066P,1,NaN,0.00
DCGS0059,1,NaN,0.00


In [ ]:
df = gabungan.copy()
df["revenue"] = df["Quantity"] * df["Price"]

KODE_NON_PRODUK = ["POST", "DOT", "M", "m", "C2", "D", "S", "BANK CHARGES",
                   "ADJUST", "ADJUST2", "AMAZONFEE", "CRUK", "B", "TEST001", "TEST002"]

jejak = []
def terapkan(data, langkah, mask_buang):
    mask_buang = mask_buang.astype(bool)
    dibuang = data[mask_buang]
    jejak.append({"langkah": langkah,
                  "baris_dibuang": len(dibuang),
                  "nilai_dibuang": round(dibuang["revenue"].sum(), 2)})
    return data[~mask_buang], dibuang

df, _ = terapkan(df, "1. Baris kembar persis", df.duplicated(keep="first"))
df, pembatalan = terapkan(df, "2. Pembatalan (awalan C), dipisahkan",
                          df["Invoice"].str.startswith("C"))
df, _ = terapkan(df, "3. Kode non-produk", df["StockCode"].isin(KODE_NON_PRODUK))
df, _ = terapkan(df, "4. Jumlah atau harga <= 0",
                 (df["Quantity"] <= 0) | (df["Price"] <= 0))

penjualan = df.reset_index(drop=True)
rekap = pd.DataFrame(jejak)
print(rekap.to_string(index=False))

assert len(gabungan) == len(penjualan) + rekap["baris_dibuang"].sum()
assert not penjualan.duplicated().any()
assert (penjualan["Quantity"] > 0).all() and (penjualan["Price"] > 0).all()
assert not penjualan["Invoice"].str.startswith("C").any()
assert not penjualan["StockCode"].isin(KODE_NON_PRODUK).any()

pelanggan = penjualan[penjualan["Customer ID"].notna()].copy()
pelanggan["Customer ID"] = pelanggan["Customer ID"].astype("int64").astype("string")

out = Path("..") / "data" / "processed"
out.mkdir(parents=True, exist_ok=True)
penjualan.to_csv(out / "penjualan.csv", index=False)
pelanggan.to_csv(out / "pelanggan.csv", index=False)
pembatalan.to_csv(out / "pembatalan.csv", index=False)
print("Tersimpan di:", out.resolve())

                             langkah  baris_dibuang  nilai_dibuang
              1. Baris kembar persis          11812       54228.42
2. Pembatalan (awalan C), dipisahkan          19104    -1462050.61
                  3. Kode non-produk           4511      671966.50
           4. Jumlah atau harga <= 0           5987           0.00
Tersimpan di: C:\Users\sarah\Documents\online-retail-customer-analysis\data\processed


In [21]:
total = penjualan["revenue"].sum()
ber_id = pelanggan["revenue"].sum()
print(f"Pendapatan bersih (semua penjualan): {total:,.2f}")
print(f"Pendapatan dari pelanggan ber ID : {ber_id:,.2f} ({ber_id / total:.1%})")
print("Pelanggan unik:", pelanggan["Customer ID"].nunique())
print("Invoice unik:", pelanggan["Invoice"].nunique())
print("Rentang tanggal:", penjualan["InvoiceDate"].min(), "->", penjualan["InvoiceDate"].max())
print()
print("Kontribusi pendapatan per negara (3 teratas):")
print((penjualan.groupby("Country")["revenue"].sum() / total)
      .sort_values(ascending=False).head(3).map("{:.1%}".format))




Pendapatan bersih (semua penjualan): 19,645,617.81
Pendapatan dari pelanggan ber ID : 17,068,582.74 (86.9%)
Pelanggan unik: 5852
Invoice unik: 36594
Rentang tanggal: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00

Kontribusi pendapatan per negara (3 teratas):
Country
United Kingdom    85.5%
EIRE               3.2%
Netherlands        2.8%
Name: revenue, dtype: str
